# ML-04 — Sibling-Page Interaction Data Contract

## 1. Grain and windows

The query table has client-content-query grain and the content table has content grain. The model
has one unordered same-client page pair per row. Evidence uses the fixed 90-day query snapshot;
movement compares previous 30 with last 30 days. This is a current diagnostic, not prediction.

In [1]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
con.sql(f"""SELECT COUNT(*) row_count,
COUNT(*)-COUNT(DISTINCT client_hash_id||'|'||content_a||'|'||content_b) duplicate_pairs,
SUM((content_a>=content_b)::INT) unordered_key_violations,
SUM((shared_query_count<2)::INT) evidence_floor_violations FROM {R}""").df()

,row_count,duplicate_pairs,unordered_key_violations,evidence_floor_violations
0,362562,0,0.0,0.0


## 2. Field roles

Features: weighted overlap, smaller-page coverage, overlap breadth, shared demand, position
proximity, visibility balance, recent movement, metadata agreement, and evidence-quality flags.

Context only: client/content hashes and dates. Excluded: hash identities as predictors, raw text,
deletion status as discovery evidence, provider/model fields, product flags, and future data.
There is no label.

In [2]:
pd.DataFrame([
("overlap and coverage","feature","relationship strength"),
("position gap and balance","feature","fragmentation structure"),
("growth_a and growth_b","feature","current movement"),
("intent and content type","feature","coarse agreement"),
("all hashes","context","join/group only"),
("is_deleted","excluded","selection only")],
columns=["fields","role","reason"])

,fields,role,reason
0,overlap and coverage,feature,relationship strength
1,position gap and balance,feature,fragmentation structure
2,growth_a and growth_b,feature,current movement
3,intent and content type,feature,coarse agreement
4,all hashes,context,join/group only
5,is_deleted,excluded,selection only


## 3. Missingness and limits

Growth is undefined when previous visible-query impressions are zero, so explicit flags replace
silent zero imputation. Rare and anonymized query detail is unavailable; measured overlap is a
lower-bound diagnostic. Meaning, text, conversions, and business purpose are absent. Actions
always mean review, never automatic merge or deletion.

In [3]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
con.sql(f"""SELECT AVG((growth_a IS NULL)::INT) growth_a_missing,
AVG((growth_b IS NULL)::INT) growth_b_missing,
AVG((main_intent_a IS NULL OR main_intent_b IS NULL)::INT) intent_missing,
AVG((word_count_a IS NULL OR word_count_b IS NULL)::INT) word_count_missing FROM {R}""").df()

,growth_a_missing,growth_b_missing,intent_missing,word_count_missing
0,0.021679,0.02153,0.00657,0.272298


## 4. Reproducibility and safety

work/scripts/build_pair_features.py aggregates remotely into a Git-ignored Parquet cache. It
prints no credentials, private queries, URLs, client names, or raw exports. Rerunners authenticate
with hf auth login. Public output contains aggregates only.